# Modify IPI and Length of cloud gaming PCAPs

### Load model

In [ ]:
from tensorflow.keras.models import load_model
from scapy.all import rdpcap, IP, Raw, wrpcap, UDP, TCP
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib
import os

# Load the model from a file
model_path = os.path.abspath("../../models/saved/DL/model10_DL.keras")
model = load_model(model_path)

WINDOW_SIZE=10

# Now you can use `model` to make predictions or evaluate on data
# Example: model.predict(X_test)

: 

### Load scaler parameters

In [ ]:
# Load the scaler
scaler_path = os.path.abspath("../../models/saved/DL/minmax_scaler_DL.joblib")
scaler = joblib.load(scaler_path)

In [ ]:
from scapy.all import rdpcap, wrpcap, IP, TCP, UDP, Raw
import numpy as np

# Function to compute IPI (Inter-Packet Interval) and Length
def compute_features(packets, idx):
    """
    Given the packets and an index `idx`, compute the required features.
    Returns the features for the model: IPI, Length, and lags.
    """
    ipi = abs(packets[idx].time - packets[idx-1].time) if idx > 0 else 0
    length = len(packets[idx])  # Length of the packet

    # Handle missing lags for the first few packets by checking the index
    lag_1_IPI = abs(float(packets[idx-1].time) - float(packets[idx-2].time)) if idx > 1 else 0
    lag_2_IPI = abs(float(packets[idx-2].time) - float(packets[idx-3].time)) if idx > 2 else 0
    lag_3_IPI = abs(float(packets[idx-3].time) - float(packets[idx-4].time)) if idx > 3 else 0
    
    lag_1_Length = float(len(packets[idx-1])) if idx > 0 else 0
    lag_2_Length = float(len(packets[idx-2])) if idx > 1 else 0
    lag_3_Length = float(len(packets[idx-3])) if idx > 2 else 0

    features = [
        float(ipi),  # IPI
        float(length),  # Length
        float(lag_1_IPI),  # lag_1_IPI
        float(lag_2_IPI),  # lag_2_IPI
        float(lag_3_IPI),  # lag_3_IPI
        float(lag_1_Length),  # lag_1_Length
        float(lag_2_Length),  # lag_2_Length
        float(lag_3_Length)   # lag_3_Length
    ]

    return features

# Function to create windows from the packet sequence
def create_packet_windows(packets, window_size):
    """
    Given a list of packets, create windows of the specified size with features for model prediction.
    """
    sequences, targets = [], []

    for i in range(len(packets) - window_size):
        # Create a window of packets (window_size = 10)
        window = packets[i:i + window_size]
        
        # Extract features for all packets in the window
        features = []
        for j in range(window_size):  # For each packet in the window
            packet_features = compute_features(window, j)
            features.append(packet_features)
        
        # Debug: Print the features for the current window
        print(f"Window {i}: Features extracted:", features)
        
        # Prepare the input for the model: (1, window_size, 8)
        input_data = np.array(features).reshape(1, window_size, 8)  # Ensure shape is (1, 10, 8)
        
        # Debug: Print the reshaped input data shape
        print(f"Window {i}: Reshaped input_data shape:", input_data.shape)
        
        # Predict the 11th packet's timestamp and length (i + window_size = i + 10)
        predicted_values = model.predict(input_data)
        predicted_timestamp = predicted_values[0][0]
        predicted_packet_len = predicted_values[0][1]

        # Store the sequence (window) and the predicted values (target)
        sequences.append(window)
        targets.append((predicted_timestamp, predicted_packet_len))

    return sequences, targets

# Load the existing pcap file
packets = rdpcap("../../testing/port80.pcap")[:20]  # Limit to first 20 packets for testing

# List to store the modified packets
modified_packets = []

# Create windows from packets for prediction
windows, targets = create_packet_windows(packets, 10)  # Assuming window_size=10

# Iterate over the windows and modify the packets based on the prediction
for i, (window, target) in enumerate(zip(windows, targets)):
    predicted_timestamp, predicted_packet_len = target
    print("Predicted:", predicted_timestamp, predicted_packet_len)
    
    # The index of the 11th packet (next packet after the 10th window)
    idx_11th_packet = i + 10

    if idx_11th_packet < len(packets):  # Ensure we're not going out of bounds
        # Get the original 11th packet
        packet = packets[idx_11th_packet]

        # Modify the packet (timestamp and length) based on the prediction
        original_timestamp = packet.time  # Preserve the original timestamp
        packet.time = predicted_timestamp  # Update the timestamp with the predicted value

        # Adjust Raw payload length to match the predicted length
        if Raw in packet:
            packet[Raw] = Raw(b"A" * int(predicted_packet_len))  # Set payload to predicted length
        else:
            packet = packet / Raw(b"A" * int(predicted_packet_len))  # Append Raw layer with predicted length

        # Recalculate the length fields
        packet[IP].len = len(packet)  # Set the IP length field manually

        # Check if the packet is UDP and recalculate the UDP length and checksum if necessary
        if UDP in packet:
            packet[UDP].len = len(packet[UDP])  # Set the UDP length field
            del packet[UDP].chksum  # Delete UDP checksum to recompute it
        elif TCP in packet:
            packet[TCP].len = len(packet[TCP])  # Set the TCP length field
            del packet[TCP].chksum  # Delete TCP checksum to recompute it

        # Recalculate the IP checksum
        del packet[IP].chksum
        packet = packet.__class__(bytes(packet))  # Rebuild the packet to recompute checksums

        # Append the modified packet to the list
        modified_packets.append(packet)

# Save the modified packets to a new pcap file
wrpcap("test_modified_fields.pcap", modified_packets)


Window 0: Features extracted: [[0.0, 54.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.004876, 60.0, 0.0, 0.0, 0.0, 54.0, 0.0, 0.0], [1.14163, 54.0, 0.004875898361206055, 0.0, 0.0, 60.0, 54.0, 0.0], [0.028773, 66.0, 1.1416301727294922, 0.004875898361206055, 0.0, 54.0, 60.0, 54.0], [1.301665, 451.0, 0.028772830963134766, 1.1416301727294922, 0.004875898361206055, 66.0, 54.0, 60.0], [0.013234, 673.0, 1.3016650676727295, 0.028772830963134766, 1.1416301727294922, 451.0, 66.0, 54.0], [4.7e-05, 66.0, 0.013234138488769531, 1.3016650676727295, 0.028772830963134766, 673.0, 451.0, 66.0], [1.1061, 451.0, 4.696846008300781e-05, 0.013234138488769531, 1.3016650676727295, 66.0, 673.0, 451.0], [0.012626, 672.0, 1.1060998439788818, 4.696846008300781e-05, 0.013234138488769531, 451.0, 66.0, 673.0], [5.2e-05, 66.0, 0.012626171112060547, 1.1060998439788818, 4.696846008300781e-05, 672.0, 451.0, 66.0]]
Window 0: Reshaped input_data shape: (1, 10, 8)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
Window 1: Features extracted: [[